# Silver Macro Transform

Thin notebook entrypoint for macro-domain Silver transforms.

Current implementation supports only the ECB branch:

- `brz_macro.raw_ecb_fx_ref_rates_daily`
- -> `slv_macro.ecb_fx_ref_rates_daily`
- rejected rows -> `slv_macro.ecb_fx_ref_rates_daily_quarantine`

The transform logic lives in `src/lakehouse`.

In [ ]:
import sys
import uuid
from pathlib import Path


def add_repo_src_to_path() -> str:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        src_dir = candidate / "src"
        if (src_dir / "lakehouse" / "__init__.py").exists():
            src_dir_str = str(src_dir)
            if src_dir_str not in sys.path:
                sys.path.insert(0, src_dir_str)
            return src_dir_str

    raise RuntimeError("Could not locate the repository src directory from the current notebook path")


REPO_SRC = add_repo_src_to_path()

from lakehouse.pipelines.silver import run_silver_ecb_fx_ref_rates_daily

spark.conf.set("spark.sql.session.timeZone", "UTC")

dbutils.widgets.text("catalog", "market_macro")
dbutils.widgets.text("source_system", "ecb")
dbutils.widgets.text("quote_currencies", "USD")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("lookback_days", "7")
dbutils.widgets.text("run_id", str(uuid.uuid4()))


In [ ]:
catalog = dbutils.widgets.get("catalog").strip() or "market_macro"
source_system = dbutils.widgets.get("source_system").strip().lower()

if source_system != "ecb":
    raise ValueError("This thin notebook currently supports only source_system='ecb'")

result = run_silver_ecb_fx_ref_rates_daily(
    spark=spark,
    raw_quote_currencies=dbutils.widgets.get("quote_currencies").strip(),
    mode=dbutils.widgets.get("mode").strip().lower(),
    start_date_raw=dbutils.widgets.get("start_date").strip(),
    end_date_raw=dbutils.widgets.get("end_date").strip(),
    lookback_days_raw=dbutils.widgets.get("lookback_days").strip(),
    run_id=dbutils.widgets.get("run_id").strip(),
    catalog=catalog,
    display_fn=display,
)

result.as_dict()
